In [2]:
!pip install xgboost lightgbm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import xgboost, lightgbm, catboost, sklearn
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print(sklearn.__version__, xgboost.__version__, lightgbm.__version__, catboost.__version__)

1.6.1 3.4.1 4.6.0 1.2.10


In [7]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
target = "BeatsPerMinute"
features = [c for c in train.columns if c not in ("id", target)]
X, y = train[features], train[target]
sample = train.sample(30_000, random_state=42)
Xs, ys = sample[features], sample[target]
Xt, yt = Xs.iloc[:15_000], ys.iloc[:15_000]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(train.shape, test.shape)
train.head()

(524164, 11) (174722, 10)


,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0,0.603610,-7.636942,0.023500,0.000005,0.000001,0.051385,0.409866,290715.6450,0.826267,147.53020
1,1,0.639451,-16.267598,0.071520,0.444929,0.349414,0.170522,0.651010,164519.5174,0.145400,136.15963
2,2,0.514538,-15.953575,0.110715,0.173699,0.453814,0.029576,0.423865,174495.5667,0.624667,55.31989
3,3,0.734463,-1.357000,0.052965,0.001651,0.159717,0.086366,0.278745,225567.4651,0.487467,147.91212
4,4,0.532968,-13.056437,0.023500,0.068687,0.000001,0.331345,0.477769,213960.6789,0.947333,89.58511


In [8]:
def cv_rmse(model, X, y, cv=kf):
    scores = -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    return scores.mean(), scores.std()

results = []

baseline_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
}
for name, model in baseline_models.items():
    mean, std = cv_rmse(model, Xs, ys)
    results.append({"model": name, "stage": "default", "rmse": mean, "std": std})
    print(name, round(mean, 4), round(std, 4))
pd.DataFrame(results)

Linear Regression 26.4514 0.0586
Decision Tree 38.2945 0.3007


,model,stage,rmse,std
0,Linear Regression,default,26.451446,0.058610
1,Decision Tree,default,38.294540,0.300699


In [9]:
advanced_models = {
    "Random Forest": RandomForestRegressor(n_estimators=50, min_samples_leaf=5, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, n_jobs=-1, random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(iterations=300, random_seed=42, verbose=0),
}
for name, model in advanced_models.items():
    mean, std = cv_rmse(model, Xs, ys)
    results.append({"model": name, "stage": "default", "rmse": mean, "std": std})
    print(name, round(mean, 4), round(std, 4))
pd.DataFrame(results)

Random Forest 26.7988 0.0873
XGBoost 28.317 0.0819
LightGBM 26.8069 0.1014
CatBoost 26.9301 0.0945


,model,stage,rmse,std
0,Linear Regression,default,26.451446,0.058610
1,Decision Tree,default,38.294540,0.300699
2,Random Forest,default,26.798812,0.087321
3,XGBoost,default,28.317030,0.081917
4,LightGBM,default,26.806926,0.101427
5,CatBoost,default,26.930121,0.094491


In [13]:
search_spaces = {
    "Decision Tree": (
        DecisionTreeRegressor(random_state=42),
        {"max_depth": randint(3, 15), "min_samples_leaf": randint(5, 200)},
    ),
    "Random Forest": (
        RandomForestRegressor(n_jobs=-1, random_state=42),
        {"n_estimators": randint(30, 80), "max_depth": randint(6, 14),
         "min_samples_leaf": randint(5, 100), "max_features": uniform(0.4, 0.6)},
    ),
    "XGBoost": (
        XGBRegressor(n_jobs=-1, random_state=42),
        {"learning_rate": loguniform(0.01, 0.2), "max_depth": randint(3, 8),
         "n_estimators": randint(100, 300)},
    ),
    "LightGBM": (
        LGBMRegressor(n_jobs=-1, random_state=42, verbose=-1),
        {"num_leaves": randint(15, 64), "min_child_samples": randint(10, 200),
         "learning_rate": loguniform(0.01, 0.2), "n_estimators": randint(100, 300)},
    ),
    "CatBoost": (
        CatBoostRegressor(random_seed=42, verbose=0),
        {"iterations": randint(100, 300), "depth": randint(4, 8),
         "l2_leaf_reg": loguniform(1, 20)},
    ),
}

best_params = {}
for name, (estimator, space) in search_spaces.items():
    search = RandomizedSearchCV(
        estimator, space, n_iter=5, cv=3,
        scoring="neg_root_mean_squared_error", random_state=42, n_jobs=1, refit=False,
    )
    search.fit(Xt, yt)
    params = {k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()}
    best_params[name] = params
    mean, std = cv_rmse(clone(estimator).set_params(**params), Xs, ys)
    results.append({"model": name, "stage": "tuned", "rmse": mean, "std": std})
    print(name, params, round(mean, 4))

Decision Tree {'max_depth': 7, 'min_samples_leaf': 107} 26.5634
Random Forest {'max_depth': 11, 'max_features': 0.4338469474162602, 'min_samples_leaf': 92, 'n_estimators': 73} 26.4734
XGBoost {'learning_rate': 0.015958237752949748, 'max_depth': 5, 'n_estimators': 174} 26.503
LightGBM {'learning_rate': 0.015341718702799927, 'min_child_samples': 140, 'n_estimators': 249, 'num_leaves': 16} 26.4903
CatBoost {'depth': 6, 'iterations': 171, 'l2_leaf_reg': 6.009974718380312} 26.4782


In [14]:
summary = (pd.DataFrame(results)
           .groupby(["model", "stage"])["rmse"].last()
           .unstack()[["default", "tuned"]]
           .sort_values("tuned"))
summary

stage,default,tuned
model,,
Random Forest,26.798812,26.473421
CatBoost,26.930121,26.478193
LightGBM,26.806926,26.490284
XGBoost,28.317030,26.502993
Decision Tree,38.294540,26.563384
Linear Regression,26.451446,NaN
